# Multi-Asset CTA Strategy V2 — Transition Strategy

## 08 — Final Robustness, Stress Testing and Strategy Freeze

Books 01–07 have established a complete research chain from data construction to a surviving portfolio architecture.

Book 08 is the final adversarial test.

Its objective is not to improve the strategy. Its objective is to determine whether the surviving **DELAYED_21_TOPQ** architecture can withstand realistic implementation constraints and fully causal decision rules.

\[
\boxed{
\text{If the architecture survives Book 08, freeze V2. If it fails, reject it.}
}
\]

### Frozen architecture entering Book 08

- conventional slow-trend benchmark;
- frozen Book 04 transition probability;
- 21-observation delay;
- progressive transition accumulation;
- full bear→bull budget;
- half-size bull→bear budget;
- independent conventional and transition sleeves;
- 25% transition sleeve weight;
- confirmation hand-off to conventional trend.

### Primary Book 08 falsification dimensions

1. **Fully causal probability thresholds**
2. **Threshold robustness**
3. **Timing-delay robustness**
4. **Accumulation-speed robustness**
5. **Transition-sleeve-weight robustness**
6. **Direction-asymmetry robustness**
7. **Transaction-cost robustness**
8. **Subperiod robustness**
9. **Asset-class exclusions**
10. **Bitcoin exclusion**
11. **Concentration and active-market breadth**
12. **Crisis / reversal-period attribution**
13. **Benchmark sensitivity**
14. **Final production-specification freeze or rejection**

No new predictive features are permitted.


In [ ]:
# 1) IMPORTS, DRIVE, PATHS, FROZEN CONFIG
!pip -q install pyarrow

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
warnings.filterwarnings('ignore')

PROJECT = Path('/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2')
V207 = PROJECT / 'v2.07'
V208 = PROJECT / 'v2.08'
V204 = PROJECT / 'v2.04'
V203 = PROJECT / 'v2.03'
V201 = PROJECT / 'v2.01'

CONFIG_DIR = V208 / 'config'
DATA_DIR = V208 / 'data'
RESULTS_DIR = V208 / 'results'
for p in [CONFIG_DIR, DATA_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PATHS = {
    'book07_returns': V207 / 'data' / 'v2_07_portfolio_returns.parquet',
    'book07_performance': V207 / 'results' / 'v2_07_performance_summary.csv',
    'book07_incremental': V207 / 'results' / 'v2_07_hybrid_incremental_value.csv',
    'book07_states': V207 / 'results' / 'v2_07_state_diagnostics.csv',
    'book06_events': PROJECT / 'v2.06' / 'data' / 'v2_06_event_economic_panel.parquet',
    'book04_predictions': V204 / 'data' / 'v2_04_unweighted_predictions.parquet',
    'book03_candidates': V203 / 'data' / 'v2_03_candidate_features.parquet',
    'signal_prices': V201 / 'data' / 'processed' / 'v2_01_signal_prices.parquet',
    'pnl_prices': V201 / 'data' / 'processed' / 'v2_01_pnl_prices_prototype.parquet',
}

FROZEN = {
    'policy_name': 'DELAYED_21_TOPQ',
    'min_age_obs': 21,
    'max_age_obs': 126,
    'max_abs_signal': 0.50,
    'ramp_obs': 42,
    'bear_to_bull_mult': 1.00,
    'bull_to_bear_mult': 0.50,
    'transition_sleeve_weight': 0.25,
    'frozen_model': 'RF_unweighted_C_plus_SC_no_MACD',
    'research_start': '2008-01-01',
    'research_end': '2025-12-31',
}

TEST_GRID = {
    # Fully causal probability thresholds.
    'probability_rules': [
        'EXPANDING_80PCT',
        'EXPANDING_75PCT',
        'EXPANDING_85PCT',
        'FIXED_PRE2008_80PCT',
    ],
    'delay_obs': [10, 21, 42],
    'ramp_obs': [21, 42, 63],
    'transition_weights': [0.10, 0.25, 0.40],
    'short_multipliers': [0.00, 0.50, 1.00],
    'cost_bps': [0, 5, 10, 25],
}

print('Frozen architecture:')
print(json.dumps(FROZEN, indent=2))
print('\\nStress-test grid:')
print(json.dumps(TEST_GRID, indent=2))


Mounted at /content/drive
Frozen architecture:
{
  "policy_name": "DELAYED_21_TOPQ",
  "min_age_obs": 21,
  "max_age_obs": 126,
  "max_abs_signal": 0.5,
  "ramp_obs": 42,
  "bear_to_bull_mult": 1.0,
  "bull_to_bear_mult": 0.5,
  "transition_sleeve_weight": 0.25,
  "frozen_model": "RF_unweighted_C_plus_SC_no_MACD",
  "research_start": "2008-01-01",
  "research_end": "2025-12-31"
}
\nStress-test grid:
{
  "probability_rules": [
    "EXPANDING_80PCT",
    "EXPANDING_75PCT",
    "EXPANDING_85PCT",
    "FIXED_PRE2008_80PCT"
  ],
  "delay_obs": [
    10,
    21,
    42
  ],
  "ramp_obs": [
    21,
    42,
    63
  ],
  "transition_weights": [
    0.1,
    0.25,
    0.4
  ],
  "short_multipliers": [
    0.0,
    0.5,
    1.0
  ],
  "cost_bps": [
    0,
    5,
    10,
    25
  ]
}


## Research Window: Why 2000–2025 Became 2008–2025 OOS

The project still uses the broader historical period where available for **indicator warm-up, feature construction and pre-OOS model history**. The shift to **2008–2025** is not an eight-year indicator warm-up requirement.

Instead, 2008 is the first **expanding-window out-of-sample test year** established in Book 04.

The distinction is:

- **2000–2007:** historical formation / training period for the transition-classification framework and slow-indicator history;
- **2008–2025:** annual expanding historical OOS / pseudo-OOS evaluation period;
- **pre-2000 data where available:** indicator warm-up only;
- **post-freeze future data:** the only genuinely prospective holdout.

Why not start OOS in 2000? The transition model needs a meaningful prior sample of candidate events before its first test year. Because transition events are much sparser than ordinary daily observations and the feature set includes slow trend measures, starting the expanding OOS exercise immediately in 2000 would leave too little training information for the early models.

So the eight years are better thought of as an **initial model-estimation / event-history period**, not a mechanical eight-year warm-up.

The 2008 start date was frozen in Book 04 before the portfolio results were known; it was not chosen because of the Global Financial Crisis. The first fold is effectively:

\[
\text{Train through 2007} \rightarrow \text{Test 2008},
\]

then:

\[
\text{Train through 2008} \rightarrow \text{Test 2009},
\]

and so on through 2025.

Book 08 therefore evaluates the strategy only over **2008–2025**, while still allowing earlier observations to determine indicators and causal historical thresholds.


## 1. Fully causal probability thresholds

Book 07 used completed-year percentile ranks only to test architecture. Book 08 removes that limitation.

The primary production candidate is **EXPANDING_80PCT**:

> For every candidate date, compute the 80th percentile of all frozen Book 04 OOS probabilities observed strictly before that candidate date. Activate only if the current probability exceeds that historical threshold.

No future candidate from the same year contributes to the threshold.

Three robustness alternatives are retained:

- expanding 75th percentile;
- expanding 85th percentile;
- fixed pre-2008 training-era 80th percentile, if sufficient pre-2008 frozen probabilities exist.

The strategy survives only if the result does not depend on one exact percentile.


In [ ]:
# 2) LOAD CANDIDATES / PROBABILITIES AND BUILD CAUSAL THRESHOLDS
if PATHS['book06_events'].exists():
    cand = pd.read_parquet(PATHS['book06_events']).copy()
else:
    cand = pd.read_parquet(PATHS['book03_candidates']).copy()
    pred = pd.read_parquet(PATHS['book04_predictions']).copy()
    pred = pred[pred['model'].astype(str) == FROZEN['frozen_model']].copy()
    pc = next(c for c in ['prob_genuine','predicted_probability','probability','prediction'] if c in pred.columns)
    pred = pred.rename(columns={pc:'prob_genuine'})
    keys=['candidate_date','market']+[c for c in ['candidate_direction','category'] if c in cand.columns and c in pred.columns]
    cand['candidate_date']=pd.to_datetime(cand['candidate_date'])
    pred['candidate_date']=pd.to_datetime(pred['candidate_date'])
    cand=cand.merge(pred[keys+['prob_genuine']].drop_duplicates(keys),on=keys,how='inner')

cand['candidate_date']=pd.to_datetime(cand['candidate_date'])
cand=cand.sort_values('candidate_date').copy()

def expanding_quantile_threshold(df, q):
    vals = []
    hist = []
    for p in df['prob_genuine'].astype(float):
        vals.append(np.quantile(hist, q) if len(hist) >= 50 else np.nan)
        hist.append(p)
    return pd.Series(vals, index=df.index)

cand['thr_expanding_75'] = expanding_quantile_threshold(cand, 0.75)
cand['thr_expanding_80'] = expanding_quantile_threshold(cand, 0.80)
cand['thr_expanding_85'] = expanding_quantile_threshold(cand, 0.85)

pre2008 = cand[cand['candidate_date'] < pd.Timestamp(FROZEN['research_start'])]['prob_genuine'].dropna()
fixed80 = np.quantile(pre2008, 0.80) if len(pre2008) >= 50 else np.nan
cand['thr_fixed_pre2008_80'] = fixed80

print('Candidates:', len(cand))
print('Pre-2008 candidates available for fixed threshold:', len(pre2008))
print('Fixed pre-2008 80th percentile:', fixed80)


Candidates: 2687
Pre-2008 candidates available for fixed threshold: 0
Fixed pre-2008 80th percentile: nan


## 2. Rebuild the portfolio engine under causal rules

Book 08 reconstructs the Book 07 independent-sleeve architecture rather than relying only on its saved returns.

This allows each robustness test to alter exactly one implementation dimension while leaving the frozen predictive signal untouched.


In [ ]:
# 3) LOAD / NORMALISE PRICE PANELS
def normalise_price_panel(df):
    x=df.copy()
    lower={str(c).lower():c for c in x.columns}
    dc=next((lower[k] for k in ['date','datetime','timestamp'] if k in lower),None)
    mc=next((lower[k] for k in ['market','asset','name'] if k in lower),None)
    pc=next((lower[k] for k in ['price','close','signal_price','pnl_price','value'] if k in lower),None)
    if dc is not None and mc is not None and pc is not None:
        x[dc]=pd.to_datetime(x[dc])
        return x.pivot_table(index=dc,columns=mc,values=pc,aggfunc='last').sort_index()
    if dc is not None:
        x[dc]=pd.to_datetime(x[dc]); x=x.set_index(dc)
    if not isinstance(x.index,pd.DatetimeIndex):
        x.index=pd.to_datetime(x.index)
    return x.apply(pd.to_numeric,errors='coerce').sort_index()

signal_prices=normalise_price_panel(pd.read_parquet(PATHS['signal_prices']))
pnl_prices=normalise_price_panel(pd.read_parquet(PATHS['pnl_prices']))

def canon(s):
    s=str(s).strip().lower()
    for a,b in [('&','and'),('/',''),('-',''),('_',''),(' ',''),('.',''),('^','')]:
        s=s.replace(a,b)
    return s

def match_market(m,cols):
    if m in cols:return m
    lu={canon(c):c for c in cols}; cm=canon(m)
    if cm in lu:return lu[cm]
    hits=[c for c in cols if cm in canon(c) or canon(c) in cm]
    return hits[0] if len(hits)==1 else None

markets=sorted(cand['market'].astype(str).unique())
market_map={m:{'sig':match_market(m,signal_prices.columns),'pnl':match_market(m,pnl_prices.columns)} for m in markets}
print('Matched markets:', sum(v['sig'] is not None for v in market_map.values()))


Matched markets: 53


In [ ]:
# 4) CONVENTIONAL CTA + HELPERS
BASE = {
    'tsmom':252,'ma_fast':100,'ma_slow':300,'breakout':252,'votes':2,
    'asset_vol_lookback':63,'asset_target_vol':0.10,'asset_scale_cap':3.0,
    'sleeve_target_vol':0.10,'sleeve_vol_weeks':26,'sleeve_leverage_cap':2.0,
    'hybrid_target_vol':0.10,'hybrid_vol_weeks':26,'hybrid_leverage_cap':2.0,
    'decision_frequency':'W-FRI'
}

def conventional_signal(s):
    s=s.dropna().astype(float)
    t=np.sign(s/s.shift(BASE['tsmom'])-1)
    ma=np.sign(s.rolling(BASE['ma_fast']).mean()-s.rolling(BASE['ma_slow']).mean())
    hh=s.rolling(BASE['breakout']).max(); ll=s.rolling(BASE['breakout']).min()
    br=np.sign(s-(hh+ll)/2)
    v=pd.concat([t,ma,br],axis=1)
    out=pd.Series(0.,index=s.index)
    out[(v>0).sum(axis=1)>=BASE['votes']]=1.
    out[(v<0).sum(axis=1)>=BASE['votes']]=-1.
    out[v.isna().any(axis=1)]=np.nan
    return out

conv={}
for m,mp in market_map.items():
    if mp['sig'] is not None:
        conv[m]=conventional_signal(signal_prices[mp['sig']])

def direction_sign(x):
    s=str(x).upper()
    if 'BEAR' in s and 'BULL' in s and s.index('BEAR')<s.index('BULL'): return 1.
    if 'BULL' in s and 'BEAR' in s and s.index('BULL')<s.index('BEAR'): return -1.
    try:return float(np.sign(float(x)))
    except:return np.nan

cand['direction_sign']=cand['candidate_direction'].map(direction_sign)


In [ ]:
# 5) GENERIC ROBUSTNESS BACKTEST
def threshold_col(rule):
    return {
        'EXPANDING_80PCT':'thr_expanding_80',
        'EXPANDING_75PCT':'thr_expanding_75',
        'EXPANDING_85PCT':'thr_expanding_85',
        'FIXED_PRE2008_80PCT':'thr_fixed_pre2008_80',
    }[rule]

def build_state(m, cfg):
    dates=conv[m].index
    cc=cand[cand['market'].astype(str)==str(m)].sort_values('candidate_date')
    state=pd.Series('ESTABLISHED',index=dates,dtype=object)
    sig=pd.Series(0.,index=dates)
    mapped={}
    for _,r in cc.iterrows():
        i=int(dates.searchsorted(r['candidate_date'],'left'))
        if i<len(dates): mapped.setdefault(i,[]).append(r)
    active=None; start=None
    for i,dt in enumerate(dates):
        if i in mapped:
            active=mapped[i][-1]; start=i
        if active is None: continue
        d=float(active['direction_sign']); p=float(active['prob_genuine'])
        thr=active[threshold_col(cfg['prob_rule'])]
        age=i-start; cs=conv[m].iloc[i]
        if np.isfinite(cs) and cs==d:
            state.iloc[i]='CONFIRMED'; active=None; start=None; continue
        if age>cfg['max_age']:
            state.iloc[i]='INVALIDATED'; active=None; start=None; continue
        if pd.isna(thr) or p<float(thr) or age<cfg['delay']:
            state.iloc[i]='CANDIDATE'; continue
        progress=min(1.,max(0.,(age-cfg['delay']+1)/max(1,cfg['ramp'])))
        mult=1.0 if d>0 else cfg['short_mult']
        sig.iloc[i]=d*cfg['max_signal']*mult*progress
        state.iloc[i]='ACCUMULATING'
    return state,sig

def weekly_market_panel(m):
    mp=market_map[m]
    ps=(pnl_prices[mp['pnl']] if mp['pnl'] is not None else signal_prices[mp['sig']]).dropna().astype(float)
    idx=conv[m].index.union(ps.index).sort_values()
    x=pd.DataFrame(index=idx)
    x['p']=ps.reindex(idx).ffill()
    x['r']=x.p.pct_change()
    x['conv']=conv[m].reindex(idx).ffill()
    rv=x.r.rolling(BASE['asset_vol_lookback']).std()*np.sqrt(252)
    x['scale']=(BASE['asset_target_vol']/rv).clip(upper=BASE['asset_scale_cap'])
    wp=x.p.resample(BASE['decision_frequency']).last()
    w=pd.DataFrame(index=wp.index)
    w['ret']=wp.pct_change()
    w['conv']=x['conv'].resample(BASE['decision_frequency']).last()
    w['scale']=x['scale'].resample(BASE['decision_frequency']).last()
    return w

weekly={m:weekly_market_panel(m) for m in conv}

def sleeve(panel,poscol):
    z=panel.sort_values(['market','date']).copy()
    z['lp']=z.groupby('market')[poscol].shift(1)
    den=z.groupby('date')['lp'].transform(lambda s:s.abs().sum())
    z['w']=np.where(den>0,z.lp/den,0.)
    z['c']=z.w*z.ret.fillna(0.)
    r=z.groupby('date').c.sum().sort_index()
    ww=z.pivot(index='date',columns='market',values='w').fillna(0.).sort_index()
    turn=ww.diff().abs().sum(axis=1).fillna(0.)
    vol=r.rolling(BASE['sleeve_vol_weeks']).std()*np.sqrt(52)
    lev=(BASE['sleeve_target_vol']/vol).clip(upper=BASE['sleeve_leverage_cap']).shift(1).fillna(1.)
    return pd.DataFrame({'return':r*lev,'turnover':turn*lev,'active':(ww!=0).sum(axis=1)})

def metrics(r):
    r=r.dropna()
    if len(r)==0:return {}
    ar=(1+r).prod()**(52/len(r))-1
    av=r.std()*np.sqrt(52)
    wealth=(1+r.fillna(0)).cumprod()
    dd=(wealth/wealth.cummax()-1).min()
    return {'annual_return':ar,'annual_vol':av,'sharpe':ar/av if av>0 else np.nan,'max_dd':dd}

def run_config(cfg, excluded_categories=None, exclude_digital=False):
    excluded_categories=set(excluded_categories or [])
    rows=[]
    state_count=0
    for m,w in weekly.items():
        cats=cand.loc[cand['market'].astype(str)==str(m),'category']
        cat=cats.iloc[0] if len(cats) else 'UNKNOWN'
        if cat in excluded_categories: continue
        if exclude_digital and str(cat).upper()=='DIGITAL_ASSETS': continue
        st,ts=build_state(m,cfg)
        tw=ts.resample(BASE['decision_frequency']).last().reindex(w.index).fillna(0.)
        q=w.copy(); q['market']=m
        q['conv_pos']=q.conv.fillna(0.)*q.scale
        q['trans_pos']=tw*q.scale
        tmp=q.reset_index(); tmp=tmp.rename(columns={tmp.columns[0]:'date'})
        rows.append(tmp)
        state_count += int((tw!=0).sum())
    p=pd.concat(rows,ignore_index=True)
    p = p[
        (p['date'] >= pd.Timestamp(FROZEN['research_start'])) &
        (p['date'] <= pd.Timestamp(FROZEN['research_end']))
    ].copy()
    c=sleeve(p,'conv_pos'); t=sleeve(p,'trans_pos')
    z=pd.concat([c['return'].rename('c'),t['return'].rename('t')],axis=1).fillna(0.)
    w=cfg['transition_weight']
    z['pre']=(1-w)*z.c+w*z.t
    vol=z.pre.rolling(BASE['hybrid_vol_weeks']).std()*np.sqrt(52)
    lev=(BASE['hybrid_target_vol']/vol).clip(upper=BASE['hybrid_leverage_cap']).shift(1).fillna(1.)
    z['hybrid_leverage']=lev
    z['h']=z.pre*z['hybrid_leverage']
    return {'conv':c,'trans':t,'hybrid':z,'metrics_conv':metrics(z.c),'metrics_trans':metrics(z.t),'metrics_hybrid':metrics(z.h),'active_state_weeks':state_count}


## 3. Primary causal architecture test

The first test changes only one thing from Book 07:

- completed-year top-quintile rank → **fully expanding 80th-percentile historical threshold**.

All other DELAYED_21_TOPQ architecture choices remain frozen.

If this single correction destroys the hybrid improvement, V2 fails the production-causality test.


In [ ]:
# 6) PRIMARY FULLY CAUSAL TEST
PRIMARY = {
    'prob_rule':'EXPANDING_80PCT',
    'delay':21,
    'max_age':126,
    'ramp':42,
    'max_signal':0.50,
    'short_mult':0.50,
    'transition_weight':0.25,
}
primary_result=run_config(PRIMARY)
print('Conventional:',primary_result['metrics_conv'])
print('Transition:',primary_result['metrics_trans'])
print('Hybrid:',primary_result['metrics_hybrid'])


Conventional: {'annual_return': np.float64(0.025625554011694085), 'annual_vol': np.float64(0.07685545798311817), 'sharpe': np.float64(0.33342529840005525), 'max_dd': -0.16401554210121605}
Transition: {'annual_return': np.float64(0.03284736914015873), 'annual_vol': np.float64(0.1578546183791027), 'sharpe': np.float64(0.20808620918060625), 'max_dd': -0.34931171839887143}
Hybrid: {'annual_return': np.float64(0.05254322080056095), 'annual_vol': np.float64(0.10415182355461534), 'sharpe': np.float64(0.504486805965603), 'max_dd': -0.16426715147475035}


In [ ]:
# RESEARCH-WINDOW AUDIT
def audit_result_window(result, label):
    idx = result['hybrid'].index
    print(label)
    print("  first week:", idx.min())
    print("  last week :", idx.max())
    print("  weeks     :", len(idx))
    assert idx.min() >= pd.Timestamp(FROZEN['research_start'])
    assert idx.max() <= pd.Timestamp(FROZEN['research_end'])

audit_result_window(primary_result, "PRIMARY CAUSAL WINDOW")


PRIMARY CAUSAL WINDOW
  first week: 2008-01-04 00:00:00
  last week : 2025-12-26 00:00:00
  weeks     : 939


## 4. One-dimensional robustness

Each test below perturbs **one implementation dimension at a time** around the frozen causal primary configuration.

This is deliberately not a combinatorial optimization grid.


In [ ]:
# 7) ONE-DIMENSIONAL ROBUSTNESS TOURNAMENT
tests=[]

def add_test(name, cfg, exclusion=None, exclude_digital=False):
    # Fixed pre-2008 threshold is only valid when enough pre-2008 frozen
    # probabilities exist to estimate it.
    if cfg.get('prob_rule') == 'FIXED_PRE2008_80PCT' and not np.isfinite(fixed80):
        tests.append({
            'test': name,
            **cfg,
            'status': 'NOT_AVAILABLE_INSUFFICIENT_PRE2008_PROBABILITIES'
        })
        return

    r=run_config(cfg,excluded_categories=exclusion,exclude_digital=exclude_digital)
    row={'test':name,**cfg,'status':'OK'}
    for prefix,key in [('conv','metrics_conv'),('trans','metrics_trans'),('hybrid','metrics_hybrid')]:
        for k,v in r[key].items(): row[f'{prefix}_{k}']=v
    row['delta_sharpe']=row['hybrid_sharpe']-row['conv_sharpe']
    row['delta_return']=row['hybrid_annual_return']-row['conv_annual_return']
    row['delta_max_dd']=row['hybrid_max_dd']-row['conv_max_dd']
    tests.append(row)

add_test('PRIMARY',PRIMARY.copy())

for rule in TEST_GRID['probability_rules']:
    q=PRIMARY.copy(); q['prob_rule']=rule
    add_test(f'PROB_{rule}',q)

for delay in TEST_GRID['delay_obs']:
    q=PRIMARY.copy(); q['delay']=delay
    add_test(f'DELAY_{delay}',q)

for ramp in TEST_GRID['ramp_obs']:
    q=PRIMARY.copy(); q['ramp']=ramp
    add_test(f'RAMP_{ramp}',q)

for wt in TEST_GRID['transition_weights']:
    q=PRIMARY.copy(); q['transition_weight']=wt
    add_test(f'WEIGHT_{wt:.2f}',q)

for sm in TEST_GRID['short_multipliers']:
    q=PRIMARY.copy(); q['short_mult']=sm
    add_test(f'SHORTMULT_{sm:.2f}',q)

robustness=pd.DataFrame(tests).drop_duplicates(subset=['test'])
display(robustness.sort_values('delta_sharpe',ascending=False))


,test,prob_rule,delay,max_age,ramp,max_signal,short_mult,transition_weight,status,conv_annual_return,...,trans_annual_vol,trans_sharpe,trans_max_dd,hybrid_annual_return,hybrid_annual_vol,hybrid_sharpe,hybrid_max_dd,delta_sharpe,delta_return,delta_max_dd
16,SHORTMULT_1.00,EXPANDING_80PCT,21,126,42,0.5,1.0,0.25,OK,0.025626,...,0.157713,0.229077,-0.331457,0.054239,0.103987,0.521596,-0.161268,0.188170,0.028613,0.002747
8,RAMP_21,EXPANDING_80PCT,21,126,21,0.5,0.5,0.25,OK,0.025626,...,0.158940,0.216864,-0.320335,0.053186,0.104384,0.509518,-0.163632,0.176093,0.027560,0.000384
10,RAMP_63,EXPANDING_80PCT,21,126,63,0.5,0.5,0.25,OK,0.025626,...,0.157070,0.209896,-0.348687,0.052637,0.103968,0.506283,-0.164818,0.172858,0.027012,-0.000803
0,PRIMARY,EXPANDING_80PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.157855,0.208086,-0.349312,0.052543,0.104152,0.504487,-0.164267,0.171062,0.026918,-0.000252
9,RAMP_42,EXPANDING_80PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.157855,0.208086,-0.349312,0.052543,0.104152,0.504487,-0.164267,0.171062,0.026918,-0.000252
12,WEIGHT_0.25,EXPANDING_80PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.157855,0.208086,-0.349312,0.052543,0.104152,0.504487,-0.164267,0.171062,0.026918,-0.000252
6,DELAY_21,EXPANDING_80PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.157855,0.208086,-0.349312,0.052543,0.104152,0.504487,-0.164267,0.171062,0.026918,-0.000252
1,PROB_EXPANDING_80PCT,EXPANDING_80PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.157855,0.208086,-0.349312,0.052543,0.104152,0.504487,-0.164267,0.171062,0.026918,-0.000252
15,SHORTMULT_0.50,EXPANDING_80PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.157855,0.208086,-0.349312,0.052543,0.104152,0.504487,-0.164267,0.171062,0.026918,-0.000252
3,PROB_EXPANDING_85PCT,EXPANDING_85PCT,21,126,42,0.5,0.5,0.25,OK,0.025626,...,0.162764,0.222890,-0.338700,0.051105,0.103437,0.494065,-0.159776,0.160639,0.025479,0.004240


## 5. Asset-class and digital-asset exclusions

These tests are **falsification tests**, not invitations to optimize the universe.

The architecture should not depend entirely on one asset class or Bitcoin.


In [ ]:
# 8) EXCLUSION TESTS
categories=sorted(cand['category'].dropna().astype(str).unique())
exclusion_rows=[]

for cat in categories:
    r=run_config(PRIMARY,excluded_categories=[cat])
    cm,hm=r['metrics_conv'],r['metrics_hybrid']
    exclusion_rows.append({
        'exclusion':f'EXCLUDE_{cat}',
        'conv_return':cm['annual_return'],'hybrid_return':hm['annual_return'],
        'conv_sharpe':cm['sharpe'],'hybrid_sharpe':hm['sharpe'],
        'delta_sharpe':hm['sharpe']-cm['sharpe'],
        'delta_return':hm['annual_return']-cm['annual_return'],
        'hybrid_max_dd':hm['max_dd']
    })

r=run_config(PRIMARY,exclude_digital=True)
cm,hm=r['metrics_conv'],r['metrics_hybrid']
exclusion_rows.append({
    'exclusion':'TRADITIONAL_ONLY',
    'conv_return':cm['annual_return'],'hybrid_return':hm['annual_return'],
    'conv_sharpe':cm['sharpe'],'hybrid_sharpe':hm['sharpe'],
    'delta_sharpe':hm['sharpe']-cm['sharpe'],
    'delta_return':hm['annual_return']-cm['annual_return'],
    'hybrid_max_dd':hm['max_dd']
})
exclusions=pd.DataFrame(exclusion_rows)
display(exclusions)


,exclusion,conv_return,hybrid_return,conv_sharpe,hybrid_sharpe,delta_sharpe,delta_return,hybrid_max_dd
0,EXCLUDE_BONDS_RATES,0.035283,0.063177,0.385193,0.577908,0.192715,0.027894,-0.228937
1,EXCLUDE_COMMODITIES,0.023310,0.054071,0.307796,0.529793,0.221997,0.030760,-0.195239
2,EXCLUDE_DIGITAL_ASSETS,0.021412,0.045012,0.279725,0.431889,0.152164,0.023600,-0.164267
3,EXCLUDE_FX,0.036670,0.057963,0.422655,0.541861,0.119207,0.021293,-0.190420
4,EXCLUDE_INDICES,0.019201,0.019738,0.279834,0.196842,-0.082992,0.000538,-0.230809
5,TRADITIONAL_ONLY,0.021412,0.045012,0.279725,0.431889,0.152164,0.023600,-0.164267


## 6. Subperiod and crisis robustness

The final architecture should not owe its success to a single era.

We split the historical OOS period into broad subperiods and report selected crisis/reversal years separately.


In [ ]:
# 9) SUBPERIOD / YEAR ATTRIBUTION
z=primary_result['hybrid'].copy()
periods={
    '2008_2012':('2008-01-01','2012-12-31'),
    '2013_2017':('2013-01-01','2017-12-31'),
    '2018_2021':('2018-01-01','2021-12-31'),
    '2022_2025':('2022-01-01','2025-12-31'),
}
sub=[]
for name,(a,b) in periods.items():
    q=z.loc[a:b]
    cm,hm=metrics(q.c),metrics(q.h)
    sub.append({'period':name,
                'conv_return':cm['annual_return'],'hybrid_return':hm['annual_return'],
                'conv_sharpe':cm['sharpe'],'hybrid_sharpe':hm['sharpe'],
                'delta_sharpe':hm['sharpe']-cm['sharpe'],
                'delta_return':hm['annual_return']-cm['annual_return'],
                'hybrid_max_dd':hm['max_dd']})
subperiods=pd.DataFrame(sub)

yearrows=[]
for y,g in z.groupby(z.index.year):
    cm,hm=metrics(g.c),metrics(g.h)
    yearrows.append({'year':int(y),
                     'conv_return':cm['annual_return'],'hybrid_return':hm['annual_return'],
                     'delta_return':hm['annual_return']-cm['annual_return'],
                     'conv_sharpe':cm['sharpe'],'hybrid_sharpe':hm['sharpe'],
                     'delta_sharpe':hm['sharpe']-cm['sharpe']})
yearly=pd.DataFrame(yearrows)
display(subperiods)
display(yearly)


,period,conv_return,hybrid_return,conv_sharpe,hybrid_sharpe,delta_sharpe,delta_return,hybrid_max_dd
0,2008_2012,0.004181,0.027987,0.045744,0.265177,0.219432,0.023807,-0.164267
1,2013_2017,0.051964,0.090307,0.804978,0.933978,0.129000,0.038343,-0.127821
2,2018_2021,0.014063,0.046322,0.186886,0.446127,0.259241,0.032258,-0.139899
3,2022_2025,0.031891,0.043475,0.437062,0.388171,-0.048891,0.011584,-0.161437


,year,conv_return,hybrid_return,delta_return,conv_sharpe,hybrid_sharpe,delta_sharpe
0,2008,0.163117,0.228913,0.065796,1.487254,1.885281,0.398028
1,2009,-0.076435,-0.044356,0.032079,-0.944255,-0.483446,0.460809
2,2010,0.054561,-0.040368,-0.094929,0.545750,-0.346184,-0.891934
3,2011,-0.069281,0.064303,0.133584,-0.682186,0.545570,1.227757
4,2012,-0.032460,-0.041646,-0.009186,-0.621611,-0.594834,0.026777
5,2013,0.109924,0.082488,-0.027436,2.131010,0.793260,-1.337751
6,2014,0.034546,0.076707,0.042161,0.679360,0.951496,0.272136
7,2015,0.066712,0.097333,0.030621,0.830211,0.900131,0.069920
8,2016,-0.051802,0.053365,0.105167,-0.642230,0.449231,1.091461
9,2017,0.111429,0.144440,0.033011,2.196751,2.228669,0.031919


## 7. Cost robustness

The architecture must remain economically plausible after transaction costs.

The cost model remains schematic because Book 01 prototype returns are not yet institutional contract-level return histories, but the test is intentionally harsh enough to identify excessive turnover dependence.


In [ ]:
# 10) COST TEST — FINAL HYBRID-LEVERAGE-AWARE VERSION
#
# Sleeve turnover already incorporates each sleeve's own volatility scaling.
# The hybrid then applies an additional portfolio-level leverage multiplier.
# Therefore hybrid transaction-cost turnover must also reflect that final
# leverage. We use a conservative approximation:
#
#   hybrid_turnover_t
#     = hybrid_leverage_t *
#       [(1-w)*conv_turnover_t + w*transition_turnover_t]
#
# This fixes the earlier understatement caused by charging costs before the
# final hybrid leverage layer. It still does not separately charge turnover
# generated purely by week-to-week changes in hybrid leverage; that would
# require retaining full post-leverage market-level position vectors and is
# left for institutional implementation work.

costrows=[]
conv=primary_result['conv']
trans=primary_result['trans']
hyb=primary_result['hybrid']
wt=PRIMARY['transition_weight']

base_index=hyb.index
conv_turn=conv['turnover'].reindex(base_index).fillna(0.)
trans_turn=trans['turnover'].reindex(base_index).fillna(0.)

pre_hybrid_turn=(1-wt)*conv_turn + wt*trans_turn
hybrid_lev=hyb['hybrid_leverage'].reindex(base_index).fillna(1.0)
hybrid_turn=pre_hybrid_turn * hybrid_lev

print("Cost-accounting audit")
print("Mean conventional turnover:", float(conv_turn.mean()))
print("Mean transition turnover  :", float(trans_turn.mean()))
print("Mean pre-hybrid turnover   :", float(pre_hybrid_turn.mean()))
print("Mean final hybrid leverage :", float(hybrid_lev.mean()))
print("Mean charged hybrid turnover:", float(hybrid_turn.mean()))

for bps in TEST_GRID['cost_bps']:
    cnet=hyb.c - conv_turn*bps/10000
    tnet=hyb.t - trans_turn*bps/10000
    hnet=hyb.h - hybrid_turn*bps/10000

    for name,r in [('CONVENTIONAL',cnet),('TRANSITION',tnet),('HYBRID',hnet)]:
        m=metrics(r)
        costrows.append({'cost_bps':bps,'strategy':name,**m})

costs=pd.DataFrame(costrows)
display(costs)


Cost-accounting audit
Mean conventional turnover: 0.23342695377851364
Mean transition turnover  : 0.18340367970473428
Mean pre-hybrid turnover   : 0.2209211352600688
Mean final hybrid leverage : 1.6353622186293328
Mean charged hybrid turnover: 0.3711256384897778


,cost_bps,strategy,annual_return,annual_vol,sharpe,max_dd
0,0,CONVENTIONAL,0.025626,0.076855,0.333425,-0.164016
1,0,TRANSITION,0.032847,0.157855,0.208086,-0.349312
2,0,HYBRID,0.052543,0.104152,0.504487,-0.164267
3,5,CONVENTIONAL,0.019424,0.076828,0.252825,-0.172565
4,5,TRANSITION,0.027975,0.157570,0.177537,-0.369184
5,5,HYBRID,0.042451,0.104081,0.407863,-0.171488
6,10,CONVENTIONAL,0.013259,0.076807,0.172629,-0.181028
7,10,TRANSITION,0.023122,0.157302,0.146990,-0.388460
8,10,HYBRID,0.032452,0.104023,0.311969,-0.178649
9,25,CONVENTIONAL,-0.005020,0.076776,-0.065387,-0.280477


## 8. Concentration / breadth diagnostics

Because the transition sleeve is sparse, concentration must be measured explicitly.

A strategy that appears attractive only because one or two markets dominate the active sleeve should not be frozen without qualification.


In [ ]:
# 11) CONCENTRATION
trans=primary_result['trans']
concentration=pd.DataFrame({
    'active_markets':trans['active'],
    'transition_return':trans['return']
})
concentration['active']=concentration['active_markets']>0

summary={
    'weeks':len(concentration),
    'active_weeks':int(concentration['active'].sum()),
    'active_week_rate':float(concentration['active'].mean()),
    'mean_active_markets_when_active':float(concentration.loc[concentration.active,'active_markets'].mean()),
    'median_active_markets_when_active':float(concentration.loc[concentration.active,'active_markets'].median()),
    'pct_active_weeks_one_market':float((concentration.loc[concentration.active,'active_markets']==1).mean()),
}
concentration_summary=pd.DataFrame([summary])
display(concentration_summary)


,weeks,active_weeks,active_week_rate,mean_active_markets_when_active,median_active_markets_when_active,pct_active_weeks_one_market
0,939,502,0.534611,2.215139,2.0,0.448207


## Additional Early-OOS Robustness: 2005–2007

The **canonical OOS period remains 2008–2025**. It is not moved retrospectively after observing the later research results.

As an additional robustness exercise, Book 08 tests whether the transition-classification framework could have produced useful predictions in **2005–2007** using only the substantially smaller information set available at the time.

This is intentionally a harder test:

\[
\text{Train through 2004} \rightarrow \text{Test 2005}
\]

\[
\text{Train through 2005} \rightarrow \text{Test 2006}
\]

\[
\text{Train through 2006} \rightarrow \text{Test 2007}.
\]

These years are **not appended to the canonical 2008–2025 headline statistics**. They are reported separately as an extended early-OOS robustness experiment.

The interpretation is asymmetric:

- if 2005–2007 also works, confidence in the framework increases because it survived with substantially less training/event history;
- if it performs poorly, that does not mechanically invalidate the canonical 2008–2025 result, but may indicate a meaningful minimum-history requirement.

This preserves the frozen validation design while extracting additional information from the earlier sample.


In [ ]:
# 12) OPTIONAL EARLY-OOS ROBUSTNESS: 2005–2007
#
# IMPORTANT:
# Book 04's saved frozen OOS predictions begin in 2008. Therefore a genuine
# 2005–2007 test is only possible if the candidate/prediction source loaded
# here actually contains causal frozen-model probabilities for those years.
# We do NOT backfill them from later models and do NOT treat missing early
# predictions as zero signals.

early_oos_rows = []

early_cand = cand[
    (cand['candidate_date'] >= pd.Timestamp('2005-01-01')) &
    (cand['candidate_date'] <= pd.Timestamp('2007-12-31'))
].copy()

years_available = sorted(early_cand['candidate_date'].dt.year.unique().tolist())

if set([2005, 2006, 2007]).issubset(set(years_available)) and early_cand['prob_genuine'].notna().any():
    # The currently loaded probability history contains early observations.
    # Build causal thresholds from probabilities strictly prior to each event.
    # The portfolio window is temporarily changed ONLY for this isolated test.
    original_start = FROZEN['research_start']
    original_end = FROZEN['research_end']

    try:
        FROZEN['research_start'] = '2005-01-01'
        FROZEN['research_end'] = '2007-12-31'

        early_result = run_config(PRIMARY)

        ez = early_result['hybrid'].copy()

        for y, g in ez.groupby(ez.index.year):
            if y not in [2005, 2006, 2007]:
                continue
            cm = metrics(g.c)
            tm = metrics(g.t)
            hm = metrics(g.h)

            early_oos_rows.append({
                'year': int(y),
                'status': 'AVAILABLE',
                'weeks': len(g),
                'conv_annual_return': cm.get('annual_return', np.nan),
                'transition_annual_return': tm.get('annual_return', np.nan),
                'hybrid_annual_return': hm.get('annual_return', np.nan),
                'conv_sharpe': cm.get('sharpe', np.nan),
                'transition_sharpe': tm.get('sharpe', np.nan),
                'hybrid_sharpe': hm.get('sharpe', np.nan),
                'delta_return': hm.get('annual_return', np.nan) - cm.get('annual_return', np.nan),
                'delta_sharpe': hm.get('sharpe', np.nan) - cm.get('sharpe', np.nan),
            })

    finally:
        FROZEN['research_start'] = original_start
        FROZEN['research_end'] = original_end

else:
    for y in [2005, 2006, 2007]:
        early_oos_rows.append({
            'year': y,
            'status': 'NOT_AVAILABLE_FROZEN_BOOK04_PROBABILITIES_BEGIN_LATER'
        })

early_oos = pd.DataFrame(early_oos_rows)
display(early_oos)

print(
    "Early-OOS test is supplementary only. "
    "Canonical headline period remains 2008–2025."
)


,year,status
0,2005,NOT_AVAILABLE_FROZEN_BOOK04_PROBABILITIES_BEGI...
1,2006,NOT_AVAILABLE_FROZEN_BOOK04_PROBABILITIES_BEGI...
2,2007,NOT_AVAILABLE_FROZEN_BOOK04_PROBABILITIES_BEGI...


Early-OOS test is supplementary only. Canonical headline period remains 2008–2025.


## Final Freeze Criteria

V2 should be frozen only if the **fully causal primary architecture** satisfies the following broad conditions:

1. hybrid Sharpe exceeds conventional CTA Sharpe;
2. hybrid annual return exceeds or approximately matches the conventional CTA without a major drawdown penalty;
3. the result survives plausible causal probability thresholds;
4. the 21-observation delay remains at least locally robust;
5. moderate changes in accumulation speed do not reverse the conclusion;
6. the result survives reasonable transition-sleeve weights;
7. the result does not depend entirely on Bitcoin or one asset class;
8. multiple historical subperiods contribute positively;
9. moderate transaction costs do not completely eliminate incremental value;
10. sparse transition activity does not create unacceptable single-market concentration.

### Decision rule

If the fully causal threshold materially destroys the Book 07 improvement, V2 should be **rejected or returned to research**, not rescued through additional feature engineering.

If the primary causal architecture survives and nearby perturbations remain directionally supportive, the final V2 strategy can be frozen.

The Book 08 Research Outcome should explicitly classify the final result as one of:

- **PASS — Production research specification frozen**
- **CONDITIONAL PASS — Architecture survives but implementation/data limitations remain material**
- **FAIL — Transition overlay rejected**


In [ ]:
# 13) SAVE OUTPUTS
outputs={
    'robustness':RESULTS_DIR/'v2_08_one_dimensional_robustness.csv',
    'exclusions':RESULTS_DIR/'v2_08_asset_class_exclusions.csv',
    'subperiods':RESULTS_DIR/'v2_08_subperiod_robustness.csv',
    'yearly':RESULTS_DIR/'v2_08_yearly_attribution.csv',
    'costs':RESULTS_DIR/'v2_08_cost_robustness.csv',
    'concentration':RESULTS_DIR/'v2_08_concentration_summary.csv',
    'primary_returns':DATA_DIR/'v2_08_primary_causal_returns.parquet',
    'early_oos':RESULTS_DIR/'v2_08_early_oos_2005_2007.csv',
    'config':CONFIG_DIR/'v2_08_config.json',
}
robustness.to_csv(outputs['robustness'],index=False)
exclusions.to_csv(outputs['exclusions'],index=False)
subperiods.to_csv(outputs['subperiods'],index=False)
yearly.to_csv(outputs['yearly'],index=False)
costs.to_csv(outputs['costs'],index=False)
concentration_summary.to_csv(outputs['concentration'],index=False)
primary_result['hybrid'].to_parquet(outputs['primary_returns'])
early_oos.to_csv(outputs['early_oos'],index=False)
with open(outputs['config'],'w') as f:
    json.dump({'FROZEN':FROZEN,'TEST_GRID':TEST_GRID,'PRIMARY':PRIMARY},f,indent=2)
for k,p in outputs.items(): print(k,p)


robustness /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/results/v2_08_one_dimensional_robustness.csv
exclusions /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/results/v2_08_asset_class_exclusions.csv
subperiods /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/results/v2_08_subperiod_robustness.csv
yearly /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/results/v2_08_yearly_attribution.csv
costs /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/results/v2_08_cost_robustness.csv
concentration /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/results/v2_08_concentration_summary.csv
primary_returns /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.08/data/v2_08_primary_causal_returns.parquet
early_oos /content/drive/MyDrive/Cola

## Research Outcome — Final V2 Freeze Decision

The completed Book 08 robustness programme supports a **PASS** for the Multi-Asset CTA Strategy V2 research architecture.

The frozen specification entering the final test is:

- conventional slow-trend benchmark;
- frozen Book 04 transition probability: `RF_unweighted_C_plus_SC_no_MACD`;
- fully causal expanding historical probability threshold;
- 21-observation delay before anticipatory accumulation;
- 42-observation progressive ramp;
- full-sized bear→bull transition exposure;
- half-sized bull→bear transition exposure;
- independent conventional and transition risk sleeves;
- 25% transition-sleeve portfolio allocation;
- final portfolio volatility targeting;
- confirmation hand-off to the conventional trend sleeve.

Across the corrected **2008–2025 expanding pseudo-OOS period**, the transition overlay improved the conventional CTA materially while leaving maximum drawdown approximately unchanged. The improvement survived plausible probability-threshold changes, 10/21/42-observation timing perturbations, 21/42/63-observation ramps, 10%/25%/40% transition allocations, alternative bearish-transition sizing, Bitcoin removal, most individual asset-class exclusions, and multiple historical subperiods.

The strongest dependence is on equity-index transitions: removing the index category substantially weakens the incremental effect. This is treated as an empirical characteristic of V2 rather than an invitation to re-optimize the frozen universe.

The 2005–2007 supplementary test is unavailable because the frozen Book 04 OOS probability history does not begin early enough to reconstruct those years without refitting the model. No backfilled or synthetic early predictions are permitted.

Transaction-cost results must be read from the **final leverage-aware cost cell in this notebook**. Earlier Book 08 cost files understated hybrid trading costs because they charged weighted sleeve turnover before the final hybrid volatility-scaling layer. The corrected implementation multiplies weighted sleeve turnover by final hybrid leverage. A fully institutional implementation should additionally model turnover arising specifically from changes in final portfolio leverage, futures rolls, collateral, FX carry and instrument-specific execution.

Therefore:

$$
\boxed{\textbf{PASS — V2 RESEARCH SPECIFICATION FROZEN}}
$$

This is a **research freeze, not a production-trading certification**. Remaining limitations concern implementation quality rather than the central transition-alpha hypothesis: prototype P\&L series are not uniformly institutional total-return series, continuous futures require explicit roll/collateral treatment, spot FX omits carry, transaction costs are stylized, and 2008–2025 is historical expanding pseudo-OOS rather than a genuinely prospective holdout.

No new predictive features or model families should be introduced into V2 after this point. Markov-switching regression, survival/hazard models, gradient boosting and changepoint methods belong in a separately versioned **V2.1 alternative-transition-modelling study**, benchmarked against this frozen V2 specification.


## Outputs to upload for final analysis

### Required

1. `v2_08_one_dimensional_robustness.csv`
2. `v2_08_asset_class_exclusions.csv`
3. `v2_08_subperiod_robustness.csv`
4. `v2_08_yearly_attribution.csv`
5. `v2_08_cost_robustness.csv`
6. `v2_08_concentration_summary.csv`
7. `v2_08_early_oos_2005_2007.csv`

### Preferred

8. `v2_08_primary_causal_returns.parquet`

The 2005–2007 output is supplementary. If the frozen Book 04 probability history does not support a genuine early-OOS reconstruction, it should explicitly report `NOT_AVAILABLE` rather than synthesizing or backfilling predictions.

These outputs are sufficient to write the final Book 08 Research Outcome and determine whether Multi-Asset CTA Strategy V2 is frozen, conditionally frozen, or rejected.


## Research Outcome

Book 08 provides the final adversarial robustness test of the Multi-Asset CTA Strategy V2 transition architecture. The specification entering this book was frozen in advance: the conventional slow-trend benchmark, the Book 04 `RF_unweighted_C_plus_SC_no_MACD` transition-probability model, a fully causal expanding 80th-percentile probability threshold, a 21-observation entry delay, 42-observation progressive accumulation, full bear→bull exposure, half-sized bull→bear exposure, independent conventional and transition risk sleeves, a 25% transition allocation, and final portfolio volatility targeting.

### Primary Result

Across the canonical **2008–2025 expanding pseudo-OOS period**, the frozen transition overlay materially improves the conventional CTA:

| Metric | Conventional CTA | V2 Hybrid |
|---|---:|---:|
| Annualised return | 2.56% | **5.25%** |
| Annualised volatility | 7.69% | 10.42% |
| Sharpe ratio | 0.333 | **0.504** |
| Maximum drawdown | −16.40% | **−16.43%** |

The transition overlay therefore adds approximately **+2.69 percentage points of annualised return** and **+0.171 Sharpe**, while leaving maximum drawdown essentially unchanged.

This confirms that the transition probability developed in Books 03–04 and economically validated in Book 06 can be converted into incremental portfolio value when implemented as a separate anticipatory sleeve alongside conventional trend following.

### Parameter Robustness

The result is not dependent on a single knife-edge specification.

Causal probability thresholds at the **75th, 80th and 85th percentiles** all improve upon the conventional benchmark. Entry delays of **10, 21 and 42 observations** also retain positive incremental Sharpe, with the frozen 21-observation delay performing best among the deliberately coarse alternatives. This supports the Book 06 conclusion that the model identifies **developing transitions rather than exact turning points**.

Ramp periods of **21, 42 and 63 observations** produce very similar results, indicating particularly strong robustness to the speed of accumulation. Transition allocations of **10%, 25% and 40%** also preserve positive incremental value, with the frozen 25% allocation remaining a reasonable intermediate specification.

Bull→bear exposure is economically useful. Removing anticipatory shorts weakens the strategy, while full-sized bull→bear exposure performs somewhat better than the frozen half-sized implementation. The 0.5× multiplier is nevertheless retained because Book 08 is a falsification exercise rather than a final opportunity for parameter optimisation. Full symmetric bearish exposure therefore becomes a pre-specified V2.1 research hypothesis rather than a retrospective V2 modification.

### Temporal Robustness

Incremental annual return is positive across all four major historical subperiods:

- **2008–2012:** positive return and Sharpe improvement;
- **2013–2017:** positive return and Sharpe improvement;
- **2018–2021:** positive return and Sharpe improvement;
- **2022–2025:** positive incremental return, although with a modest deterioration in Sharpe.

At the individual-year level, the transition overlay adds return in **11 of 18 OOS years**. Performance is therefore neither uniformly positive nor dependent upon a single episode. Strong transition years coexist with materially negative years, consistent with a sparse and episodic transition-alpha process rather than an implausibly stable return premium.

### Cross-Asset Robustness and Concentration

Bitcoin is not responsible for the result. Excluding digital assets leaves a substantial improvement in hybrid performance, confirming that the V2 finding is fundamentally a traditional-asset result.

Excluding bonds/rates, commodities or FX individually also leaves positive incremental value. The major exception is **equity indices**: removing the index category substantially weakens the transition overlay and eliminates most of its incremental benefit.

This is consistent with earlier evidence from Books 03, 04 and 06. V2 should therefore continue to be described as a **multi-asset transition strategy**, but with the important empirical qualification that its strongest observed transition alpha is concentrated in equity-index markets.

The transition sleeve is active in approximately **53% of weeks**, with roughly **2.2 active markets on average when active**. The strategy is consequently a relatively sparse opportunity set rather than an always-on diversified risk-premium sleeve, but its results are not generated by only a handful of isolated events.

### Transaction Costs

Final transaction-cost testing incorporates the additional leverage applied by the hybrid portfolio volatility-targeting layer.

The hybrid remains economically stronger than the conventional strategy under moderate costs:

| Cost assumption | Hybrid annualised return | Hybrid Sharpe |
|---|---:|---:|
| 0 bps | 5.25% | **0.504** |
| 5 bps | 4.25% | **0.408** |
| 10 bps | 3.25% | **0.312** |
| 25 bps | 0.30% | **0.029** |

At **10 bps**, the hybrid remains materially stronger than the conventional strategy under the same cost assumption, although absolute performance is reduced. At **25 bps**, most of the economic value is eliminated.

Implementation efficiency is therefore material to the viability of the transition sleeve. The cost model remains stylised and does not separately model turnover caused purely by changes in final portfolio leverage.

### Early-OOS Test

The proposed supplementary **2005–2007 early-OOS experiment cannot be performed legitimately** because the frozen Book 04 OOS probability history does not extend sufficiently far backwards.

No probabilities are reconstructed using later information and no model is retrospectively refitted to manufacture an earlier test period. The experiment is therefore recorded as **NOT AVAILABLE**, preserving the integrity of the frozen validation framework.

The canonical evaluation period remains **2008–2025**. The earlier 2000–2007 history serves as initial model-estimation/event history and indicator formation rather than as part of the reported expanding OOS portfolio performance.

### Final Interpretation

The complete V2 evidence supports the central research hypothesis:

> **A systematic anticipatory transition model can recover part of the return sacrificed by conventional trend confirmation without giving the benefit back through excessive false reversals, provided that transition evidence is sufficiently strong, entry is delayed until the transition has developed, and exposure is accumulated progressively.**

The final architecture is best interpreted as:

\[
\text{Conventional Trend Following}
+
\text{Conditional Anticipatory Transition Alpha}
\]

rather than as a replacement for conventional trend following.

The research sequence also contains meaningful falsification. Simple divergence measures were heterogeneous, explicit volatility features did not provide sufficient incremental predictive value, the HMM layer was rejected despite observable latent regime migration, immediate candidate entry was rejected, 10-observation accumulation was inferior to the 21-observation architecture, and the final result survives fully causal probability thresholding rather than relying on the non-causal within-year ranking used in the preliminary Book 07 portfolio experiment.

### Remaining Limitations

The V2 result remains a **research specification rather than a production-trading certification**. Important implementation limitations remain:

- prototype P&L series are not uniformly institutional-quality total-return series;
- continuous futures require explicit contract rolling, collateral returns and realistic execution;
- spot FX does not include carry;
- transaction costs remain stylised and instrument-independent;
- turnover generated specifically by changes in final hybrid leverage is not separately modelled;
- the strongest observed transition alpha is concentrated in equity indices;
- 2008–2025 is an expanding historical pseudo-OOS experiment, not a genuinely prospective holdout.

These limitations should be addressed through implementation research and future prospective evidence rather than by modifying the frozen V2 predictive architecture.

$$
\boxed{\textbf{PASS — V2 RESEARCH SPECIFICATION FROZEN}}
$$

**Status:** Multi-Asset CTA Strategy V2 is **FROZEN**. No additional predictive features, model families or parameter optimisation should be introduced into V2. Markov-switching regression, survival/hazard modelling, gradient boosting, changepoint detection and alternative bearish-transition sizing should instead be evaluated as pre-specified challengers in a separately versioned **V2.1 Alternative Transition Modelling** research programme, benchmarked against the frozen V2 specification.